# Retrieval Diversity and Redundancy in RAG
A clean end-to-end notebook for studying relevance, redundancy, and coverage in retrieval for RAG.

This version rewrites the full workflow from scratch, starting with:
1. downloading the 2019 and 2020 TREC Deep Learning query sets from MS MARCO,
2. inspecting what a query and graded relevance judgments look like,
3. building a frozen evaluation set,
4. constructing a judged-document candidate pool,
5. building dense and sparse retrieval representations,
6. implementing baseline and diversity-aware retrieval methods,
7. evaluating both ranking quality and coverage efficiency.

The notebook is organized so that another researcher can read it top to bottom and understand both the experimental setup and the code.

## 0. Research framing

**Goal.** Retrieve a set of documents that covers the meaning of a query while minimizing redundancy.

**Main question.** Can diversity-aware retrieval methods select a smaller, less redundant set of documents than standard retrieval baselines without losing too much useful information?

**Methods implemented in this notebook**
- Dense retrieval
- Sparse retrieval using standard BM25
- Dense-sparse score fusion
- Hybrid pricing
- Hybrid MMR
- Dual residual tracking

**Important design decision for dual residual in this rewrite**
- The sparse side stays close to standard sparse retrieval.
- We do **not** replace sparse retrieval with raw token-overlap coverage scoring.
- Instead, dual residual uses a **weighted BM25-style query**, where query term weights decay over time as their lexical demand is covered.
- Stopping is based on a **combined dense + sparse residual exhaustion signal**, not only a sparse threshold.

## 1. Optional installs

Uncomment and run if the environment does not already have the required libraries.

In [1]:
# %pip install -q ir_datasets sentence-transformers rank_bm25 scikit-learn pandas matplotlib tqdm

## 2. Imports

In [2]:
import json
import math
import os
import pickle
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import product
from typing import Dict, List, Tuple, Iterable, Optional

import ir_datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

/users/jacobr1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuration

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASETS = [
    "msmarco-passage/trec-dl-2019",
    "msmarco-passage/trec-dl-2020",
]

CACHE_DIR = "rag_diversity_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 10

# Retrieval defaults
DEFAULT_ALPHA = 0.75
DEFAULT_LAMBDA = 0.30
DEFAULT_ETA = 0.35

# Dual residual defaults
DEFAULT_BETA_STOP = 0.50
DEFAULT_MIN_COMBINED_RESIDUAL_FRAC = 0.10
DEFAULT_MIN_COMBINED_SCORE = 0.10

# Coverage thresholds for efficiency metrics
SEMANTIC_THRESHOLDS = [0.80, 0.90, 0.95]
SPARSE_THRESHOLDS = [0.80, 0.90, 0.95]

# Candidate pool controls
MAX_CANDIDATE_DOCS = None   # set to an integer if you want to subsample for faster experiments

# Export controls
RESULTS_CSV = os.path.join(CACHE_DIR, "method_results.csv")
SWEEP_CSV = os.path.join(CACHE_DIR, "dual_residual_sweep.csv")

print("Cache directory:", CACHE_DIR)

Cache directory: rag_diversity_cache


## 4. Load the 2019 and 2020 TREC-DL datasets

These datasets give:
- queries
- qrels with graded relevance labels
- access to the MS MARCO passage store through `docs_store()`

In [4]:
datasets = [ir_datasets.load(name) for name in DATASETS]
datasets

[Dataset(id='msmarco-passage/trec-dl-2019', provides=['docs', 'queries', 'qrels', 'scoreddocs']),
 Dataset(id='msmarco-passage/trec-dl-2020', provides=['docs', 'queries', 'qrels', 'scoreddocs'])]

## 5. Inspect what a query and graded judgments look like

This is the first sanity check in the notebook.

In [5]:
def load_queries_and_qrels(dataset):
    queries = {q.query_id: q.text for q in dataset.queries_iter()}
    qrels = defaultdict(dict)
    for qr in dataset.qrels_iter():
        qrels[qr.query_id][qr.doc_id] = qr.relevance
    return queries, dict(qrels)

all_queries = {}
all_qrels = {}
for ds in datasets:
    q, r = load_queries_and_qrels(ds)
    all_queries.update(q)
    all_qrels.update(r)

sample_qid = sorted(all_qrels.keys())[0]
print("Sample query id:", sample_qid)
print("Sample query text:", all_queries[sample_qid])
print("\nSample qrels for this query:")
print(all_qrels[sample_qid])

Sample query id: 1030303
Sample query text: who is aziz hashim

Sample qrels for this query:
{'1038342': 0, '1042969': 0, '1044043': 0, '1056584': 0, '1070133': 0, '1070134': 0, '1086238': 0, '1104408': 0, '1161432': 0, '1161439': 0, '1166176': 0, '1172906': 0, '1207443': 0, '1227100': 0, '1269099': 0, '1292821': 0, '1292822': 0, '1305520': 0, '1305521': 0, '1305528': 0, '1358683': 0, '1369019': 0, '1376556': 0, '1376558': 0, '1397758': 0, '1402779': 0, '1451846': 0, '1451848': 0, '1451850': 0, '1451851': 0, '1462522': 0, '1492373': 0, '1538148': 0, '1538153': 0, '163871': 0, '1666520': 0, '1777312': 0, '1784767': 0, '1811800': 0, '1813557': 0, '1813561': 0, '1905988': 0, '196026': 0, '2001450': 0, '2057929': 0, '2146415': 0, '2194939': 0, '2205078': 0, '2217415': 0, '2217635': 0, '22484': 0, '226464': 0, '2484376': 0, '2488127': 0, '2563261': 0, '2597066': 0, '2620119': 0, '2628843': 0, '2628844': 0, '2712400': 0, '2740876': 0, '2782702': 0, '288491': 0, '292060': 0, '309441': 0, '310

If you want to inspect a random query with graded labels, run the next cell a few times.

In [6]:
import random

random_qid = random.choice(list(all_qrels.keys()))
print("Random query id:", random_qid)
print("Query:", all_queries[random_qid])
print("Number of judged docs:", len(all_qrels[random_qid]))
print("Top judged docs by grade:")
for doc_id, rel in sorted(all_qrels[random_qid].items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(doc_id, "->", rel)

Random query id: 1106979
Query: define pareto chart in statistics
Number of judged docs: 157
Top judged docs by grade:
3290649 -> 3
8190140 -> 3
1039252 -> 2
2306620 -> 2
2375312 -> 2
2884219 -> 2
3290646 -> 2
3290651 -> 2
348054 -> 2
348057 -> 2


## 6. Build the frozen evaluation set

We keep only queries with at least one relevant document.
That gives a stable query set for consistent benchmarking.

In [7]:
def build_frozen_query_set(queries: Dict[str, str], qrels: Dict[str, Dict[str, int]]):
    queries_frozen = {}
    qrels_frozen = {}
    for qid, text in queries.items():
        judged = qrels.get(qid, {})
        if any(rel > 0 for rel in judged.values()):
            queries_frozen[qid] = text
            qrels_frozen[qid] = judged
    return queries_frozen, qrels_frozen

queries_frozen, qrels_frozen = build_frozen_query_set(all_queries, all_qrels)
print("Frozen query count:", len(queries_frozen))
print("Example query id:", next(iter(queries_frozen)))

Frozen query count: 97
Example query id: 156493


## 7. Build the judged-document candidate pool

To keep experiments controlled and reproducible, we take the union of all judged documents
appearing in the frozen qrels, then fetch the passage text for those docs.

In [8]:
docs_store = datasets[0].docs_store()

candidate_doc_ids = sorted({doc_id for qid in qrels_frozen for doc_id in qrels_frozen[qid]})
if MAX_CANDIDATE_DOCS is not None:
    candidate_doc_ids = candidate_doc_ids[:MAX_CANDIDATE_DOCS]

doc_text = {}
missing_doc_ids = []

for doc_id in tqdm(candidate_doc_ids, desc="Fetching judged passages"):
    doc = docs_store.get(doc_id)
    if doc is None:
        missing_doc_ids.append(doc_id)
        continue
    text = getattr(doc, "text", None)
    if text is None:
        text = str(doc)
    doc_text[doc_id] = text

candidate_doc_ids = [d for d in candidate_doc_ids if d in doc_text]
doc_index = {doc_id: i for i, doc_id in enumerate(candidate_doc_ids)}

print("Candidate docs loaded:", len(candidate_doc_ids))
print("Missing docs:", len(missing_doc_ids))

Fetching judged passages: 100%|█████████| 20349/20349 [00:08<00:00, 2513.39it/s]

Candidate docs loaded: 20349
Missing docs: 0


## 8. Inspect a judged document example

In [9]:
sample_doc_id = next(iter(doc_text))
print("Sample doc id:", sample_doc_id)
print(doc_text[sample_doc_id][:1000])

Sample doc id: 1000485
Pseudobulbar palsy is characterized by the inability to control facial muscles, including the tongue. This may manifest as: 1  dysarthria (slowed or slurred speech). 2  dysphagia (difficulty swallowing).seudobulbar palsy is characterized by the inability to control facial muscles, including the tongue. This may manifest as: 1  dysarthria (slowed or slurred speech). 2  dysphagia (difficulty swallowing).


## 9. Tokenization helpers

In [10]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")

def simple_tokenize(text: str) -> List[str]:
    return TOKEN_PATTERN.findall(text.lower())

tokenized_docs = [simple_tokenize(doc_text[doc_id]) for doc_id in candidate_doc_ids]
doc_term_freqs = [Counter(tokens) for tokens in tokenized_docs]
doc_lengths = np.array([len(tokens) for tokens in tokenized_docs], dtype=float)
avg_doc_len = float(doc_lengths.mean()) if len(doc_lengths) else 0.0

print("Average tokenized document length:", round(avg_doc_len, 2))

Average tokenized document length: 56.9


## 10. Standard sparse retrieval: BM25

This is the normal sparse baseline.

We will also reuse BM25 components inside dual residual, but with **residual query weights**.
The document side remains standard BM25-style term-frequency scoring.

In [11]:
bm25 = BM25Okapi(tokenized_docs)
print("BM25 index built over", len(tokenized_docs), "documents")

BM25 index built over 20349 documents


## 11. Dense representation

We encode all candidate documents once, normalize them, and keep them in memory.

In [12]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

doc_embedding_cache = os.path.join(CACHE_DIR, "doc_embeddings.npy")
if os.path.exists(doc_embedding_cache):
    doc_embeddings = np.load(doc_embedding_cache)
else:
    texts = [doc_text[doc_id] for doc_id in candidate_doc_ids]
    doc_embeddings = embed_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(doc_embedding_cache, doc_embeddings)

print("Doc embeddings shape:", doc_embeddings.shape)

Loading weights: 100%|█| 103/103 [00:00<00:00, 925.49it/s, Materializing param=p
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Doc embeddings shape: (20349, 384)


## 12. Query embedding helper

In [13]:
def embed_query(query_text: str) -> np.ndarray:
    return embed_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

## 13. Core scoring helpers

In [14]:
def normalize_scores(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return scores
    lo = np.min(scores)
    hi = np.max(scores)
    if hi - lo < 1e-12:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)

def dense_scores_for_query(query_text: str) -> np.ndarray:
    q_emb = embed_query(query_text)
    return np.asarray(doc_embeddings @ q_emb, dtype=float)

def sparse_scores_for_query(query_text: str) -> np.ndarray:
    q_tokens = simple_tokenize(query_text)
    return np.array(bm25.get_scores(q_tokens), dtype=float)

def get_relevance_grade(qid: str, doc_id: str) -> int:
    return int(qrels_frozen.get(qid, {}).get(doc_id, 0))

def upper_triangle_mean(sim_matrix: np.ndarray) -> float:
    n = sim_matrix.shape[0]
    if n < 2:
        return 0.0
    iu = np.triu_indices(n, k=1)
    return float(sim_matrix[iu].mean())

## 14. Weighted BM25 for dual residual

This is the key change.

Instead of replacing sparse retrieval with simple token overlap, we keep the sparse side close to normal BM25.
The difference is that the query now has **residual token weights**.

For a weighted query:
- each query token contributes according to its remaining residual weight
- document scoring still uses BM25-style tf, idf, and document-length normalization

In [15]:
# Extract BM25 internals
BM25_K1 = bm25.k1
BM25_B = bm25.b
BM25_IDF = bm25.idf

def weighted_bm25_scores_from_residual(sparse_residual: Counter) -> np.ndarray:
    scores = np.zeros(len(candidate_doc_ids), dtype=float)
    if not sparse_residual:
        return scores

    for tok, q_weight in sparse_residual.items():
        idf = BM25_IDF.get(tok, 0.0)
        if abs(idf) < 1e-12:
            continue

        for i, tf_counter in enumerate(doc_term_freqs):
            tf = tf_counter.get(tok, 0)
            if tf == 0:
                continue
            denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * (doc_lengths[i] / avg_doc_len))
            scores[i] += q_weight * idf * (tf * (BM25_K1 + 1.0) / denom)

    return scores

def build_sparse_query_weights(query_text: str) -> Counter:
    return Counter(simple_tokenize(query_text))

def update_sparse_residual(doc_text_value: str, sparse_residual: Counter, decay: float = 1.0) -> Counter:
    doc_tokens = set(simple_tokenize(doc_text_value))
    new_residual = Counter(sparse_residual)
    for tok in list(new_residual.keys()):
        if tok in doc_tokens:
            new_residual[tok] = max(0.0, new_residual[tok] - decay)
            if new_residual[tok] <= 0:
                del new_residual[tok]
    return new_residual

## 15. Retrieval methods

In [16]:
def retrieve_dense(query_text: str, k: int = TOP_K) -> List[str]:
    scores = dense_scores_for_query(query_text)
    top_idx = np.argsort(-scores)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_sparse(query_text: str, k: int = TOP_K) -> List[str]:
    scores = sparse_scores_for_query(query_text)
    top_idx = np.argsort(-scores)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_fusion(query_text: str, alpha: float = DEFAULT_ALPHA, k: int = TOP_K) -> List[str]:
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    fused = alpha * dense_scores + (1.0 - alpha) * sparse_scores
    top_idx = np.argsort(-fused)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_hybrid_mmr(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    lambda_div: float = DEFAULT_LAMBDA,
    k: int = TOP_K,
) -> List[str]:
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    relevance = alpha * dense_scores + (1.0 - alpha) * sparse_scores

    selected = []
    remaining = set(range(len(candidate_doc_ids)))

    while remaining and len(selected) < k:
        best_i = None
        best_score = -1e18
        for i in remaining:
            if not selected:
                score = relevance[i]
            else:
                sim_to_selected = np.max(doc_embeddings[selected] @ doc_embeddings[i])
                score = relevance[i] - lambda_div * float(sim_to_selected)
            if score > best_score:
                best_score = score
                best_i = i
        selected.append(best_i)
        remaining.remove(best_i)

    return [candidate_doc_ids[i] for i in selected]

def retrieve_hybrid_pricing(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    lambda_div: float = DEFAULT_LAMBDA,
    k: int = TOP_K,
) -> List[str]:
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    relevance = alpha * dense_scores + (1.0 - alpha) * sparse_scores

    selected = []
    remaining = set(range(len(candidate_doc_ids)))

    while remaining and len(selected) < k:
        best_i = None
        best_score = -1e18
        for i in remaining:
            redundancy = 0.0
            if selected:
                redundancy = float(np.mean(doc_embeddings[selected] @ doc_embeddings[i]))
            score = relevance[i] - lambda_div * redundancy
            if score > best_score:
                best_score = score
                best_i = i
        selected.append(best_i)
        remaining.remove(best_i)

    return [candidate_doc_ids[i] for i in selected]

## 16. Dual residual tracking

This is the main method rewrite.

### Dense side
We maintain:
- a dense residual direction `q_dense_dir`
- a scalar `dense_mass_remaining`

After selecting a document with embedding `d`, we:
1. measure how much of the current semantic direction it covers,
2. subtract that covered direction,
3. renormalize the direction,
4. separately decay the scalar semantic mass so we still have an interpretable stopping signal.

### Sparse side
We maintain residual token weights over the query.
The sparse score is a **weighted BM25 score**, not a raw token-overlap count.

### Stopping
We compute:
- `dense_residual_frac`
- `sparse_residual_frac`

Then combine them:

`combined_residual_frac = beta_stop * dense_residual_frac + (1 - beta_stop) * sparse_residual_frac`

We stop when:
- combined residual demand is small enough, or
- the best remaining document under the residual objective is too weak, or
- we reach `k`

In [17]:
def retrieve_dual_residual(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    eta: float = DEFAULT_ETA,
    sparse_decay: float = 1.0,
    beta_stop: float = DEFAULT_BETA_STOP,
    min_combined_residual_frac: float = DEFAULT_MIN_COMBINED_RESIDUAL_FRAC,
    min_combined_score: Optional[float] = DEFAULT_MIN_COMBINED_SCORE,
    k: int = TOP_K,
    return_debug: bool = False,
):
    q_dense_dir = embed_query(query_text).copy()
    dense_mass_remaining = 1.0

    sparse_residual = build_sparse_query_weights(query_text)
    initial_sparse_mass = float(sum(sparse_residual.values())) if sparse_residual else 1.0

    selected = []
    remaining = set(range(len(candidate_doc_ids)))
    history = []

    while remaining and len(selected) < k:
        dense_scores = doc_embeddings @ q_dense_dir
        dense_scores = normalize_scores(dense_scores)

        sparse_scores = weighted_bm25_scores_from_residual(sparse_residual)
        sparse_scores = normalize_scores(sparse_scores)

        combined = alpha * dense_scores + (1.0 - alpha) * sparse_scores

        remaining_list = list(remaining)
        best_i = remaining_list[int(np.argmax(combined[remaining_list]))]
        best_score = float(combined[best_i])

        sparse_mass_remaining = float(sum(sparse_residual.values()))
        sparse_residual_frac = sparse_mass_remaining / initial_sparse_mass if initial_sparse_mass > 0 else 0.0
        dense_residual_frac = dense_mass_remaining
        combined_residual_frac = beta_stop * dense_residual_frac + (1.0 - beta_stop) * sparse_residual_frac

        if len(selected) > 0:
            if combined_residual_frac < min_combined_residual_frac:
                break
            if min_combined_score is not None and best_score < min_combined_score:
                break

        selected.append(best_i)
        remaining.remove(best_i)

        # Dense residual update
        d = doc_embeddings[best_i]
        coverage = max(0.0, float(q_dense_dir @ d))

        q_dense_dir = q_dense_dir - eta * coverage * d
        norm = np.linalg.norm(q_dense_dir)
        if norm > 1e-12:
            q_dense_dir = q_dense_dir / norm

        dense_mass_remaining = max(0.0, dense_mass_remaining * (1.0 - eta * coverage))

        # Sparse residual update
        sparse_residual = update_sparse_residual(
            doc_text[candidate_doc_ids[best_i]],
            sparse_residual,
            decay=sparse_decay,
        )

        history.append({
            "step": len(selected),
            "doc_id": candidate_doc_ids[best_i],
            "best_score": best_score,
            "dense_residual_frac": dense_mass_remaining,
            "sparse_residual_frac": float(sum(sparse_residual.values())) / initial_sparse_mass if initial_sparse_mass > 0 else 0.0,
        })

    selected_doc_ids = [candidate_doc_ids[i] for i in selected]

    if return_debug:
        return selected_doc_ids, history
    return selected_doc_ids

## 17. Evaluation metrics

We evaluate both ranking-style and coverage-style metrics.

In [18]:
def dcg_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    score = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids[:k], start=1):
        rel = get_relevance_grade(qid, doc_id)
        gain = (2 ** rel) - 1
        discount = 1.0 / math.log2(rank + 1)
        score += gain * discount
    return score

def idcg_at_k(qid: str, k: int = TOP_K) -> float:
    rels = sorted(qrels_frozen[qid].values(), reverse=True)[:k]
    score = 0.0
    for rank, rel in enumerate(rels, start=1):
        gain = (2 ** rel) - 1
        discount = 1.0 / math.log2(rank + 1)
        score += gain * discount
    return score

def ndcg_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    denom = idcg_at_k(qid, k)
    if denom == 0:
        return 0.0
    return dcg_at_k(retrieved_doc_ids, qid, k) / denom

def irrelevant_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> int:
    return sum(1 for doc_id in retrieved_doc_ids[:k] if get_relevance_grade(qid, doc_id) <= 0)

def hit_grade3(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> int:
    return int(any(get_relevance_grade(qid, doc_id) == 3 for doc_id in retrieved_doc_ids[:k]))

def redundancy_at_k(retrieved_doc_ids: List[str], k: int = TOP_K) -> float:
    ids = retrieved_doc_ids[:k]
    if len(ids) < 2:
        return 0.0
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in ids])
    sim = emb @ emb.T
    return upper_triangle_mean(sim)

## 18. Coverage metrics

The current judged candidate pool makes pure coverage difficult to define perfectly, but we can still build useful proxies.

### Semantic coverage
For each relevant judged document for a query, compute its maximum cosine similarity to the selected set.
Then average those maxima, weighting by relevance grade.

This measures how well the selected set semantically covers the judged relevant set.

### Sparse coverage
Take the original query token multiset.
Measure the fraction of query token mass that appears in at least one selected document, counting repeated tokens using the original query counts.

### Efficiency
How many retrieved documents are needed to exceed given semantic or sparse coverage thresholds?

In [19]:
def relevant_doc_ids_for_query(qid: str) -> List[str]:
    return [doc_id for doc_id, rel in qrels_frozen[qid].items() if rel > 0 and doc_id in doc_index]

def semantic_coverage(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    selected = [doc_id for doc_id in retrieved_doc_ids[:k] if doc_id in doc_index]
    relevant = relevant_doc_ids_for_query(qid)

    if not selected or not relevant:
        return 0.0

    selected_emb = np.vstack([doc_embeddings[doc_index[d]] for d in selected])
    relevant_emb = np.vstack([doc_embeddings[doc_index[d]] for d in relevant])

    sim = relevant_emb @ selected_emb.T
    max_sim = sim.max(axis=1)

    weights = np.array([qrels_frozen[qid][d] for d in relevant], dtype=float)
    if weights.sum() <= 0:
        return float(max_sim.mean())
    return float(np.average(max_sim, weights=weights))

def sparse_query_coverage(retrieved_doc_ids: List[str], query_text: str, k: int = TOP_K) -> float:
    query_weights = build_sparse_query_weights(query_text)
    if not query_weights:
        return 0.0

    covered = Counter()
    for doc_id in retrieved_doc_ids[:k]:
        doc_tokens = set(simple_tokenize(doc_text[doc_id]))
        for tok in query_weights:
            if tok in doc_tokens:
                covered[tok] = query_weights[tok]

    covered_mass = float(sum(covered.values()))
    total_mass = float(sum(query_weights.values()))
    return covered_mass / total_mass if total_mass > 0 else 0.0

def docs_needed_for_semantic_coverage(retrieved_doc_ids: List[str], qid: str, threshold: float) -> Optional[int]:
    for j in range(1, len(retrieved_doc_ids) + 1):
        if semantic_coverage(retrieved_doc_ids[:j], qid, k=j) >= threshold:
            return j
    return None

def docs_needed_for_sparse_coverage(retrieved_doc_ids: List[str], query_text: str, threshold: float) -> Optional[int]:
    for j in range(1, len(retrieved_doc_ids) + 1):
        if sparse_query_coverage(retrieved_doc_ids[:j], query_text, k=j) >= threshold:
            return j
    return None

## 19. Unified evaluation wrapper

In [20]:
def evaluate_single_query(qid: str, method_name: str, retrieved_doc_ids: List[str], query_text: str, k: int = TOP_K) -> Dict:
    row = {
        "qid": qid,
        "method": method_name,
        "k": k,
        "nDCG@k": ndcg_at_k(retrieved_doc_ids, qid, k),
        "irrelevant@k": irrelevant_at_k(retrieved_doc_ids, qid, k),
        "redundancy@k": redundancy_at_k(retrieved_doc_ids, k),
        "hit_grade3@k": hit_grade3(retrieved_doc_ids, qid, k),
        "semantic_coverage@k": semantic_coverage(retrieved_doc_ids, qid, k),
        "sparse_coverage@k": sparse_query_coverage(retrieved_doc_ids, query_text, k),
        "docs_retrieved": len(retrieved_doc_ids[:k]),
    }
    for thr in SEMANTIC_THRESHOLDS:
        row[f"docs_to_semantic_{thr:.2f}"] = docs_needed_for_semantic_coverage(retrieved_doc_ids[:k], qid, thr)
    for thr in SPARSE_THRESHOLDS:
        row[f"docs_to_sparse_{thr:.2f}"] = docs_needed_for_sparse_coverage(retrieved_doc_ids[:k], query_text, thr)
    return row

def evaluate_method_over_queries(method_name: str, retrieve_fn, qids: Optional[List[str]] = None, k: int = TOP_K, verbose: bool = True) -> pd.DataFrame:
    if qids is None:
        qids = list(queries_frozen.keys())

    rows = []
    iterator = tqdm(qids, desc=f"Evaluating {method_name}") if verbose else qids
    for qid in iterator:
        query_text = queries_frozen[qid]
        retrieved_doc_ids = retrieve_fn(query_text, k=k)
        rows.append(evaluate_single_query(qid, method_name, retrieved_doc_ids, query_text, k=k))
    return pd.DataFrame(rows)

## 20. Quick spot-check on one query

In [21]:
test_qid = random.choice(list(queries_frozen.keys()))
test_query = queries_frozen[test_qid]
print("Test qid:", test_qid)
print("Query:", test_query)

for name, fn in [
    ("dense", retrieve_dense),
    ("sparse_bm25", retrieve_sparse),
    ("fusion", lambda q, k=TOP_K: retrieve_fusion(q, alpha=0.75, k=k)),
    ("dual_residual", lambda q, k=TOP_K: retrieve_dual_residual(q, alpha=0.75, eta=0.35, sparse_decay=1.0, beta_stop=0.5, min_combined_residual_frac=0.10, min_combined_score=0.10, k=k)),
]:
    docs = fn(test_query, k=TOP_K)
    print("\nMethod:", name)
    print("Top docs:", docs[:5])
    print("nDCG@10:", round(ndcg_at_k(docs, test_qid, 10), 4))
    print("hit_grade3@10:", hit_grade3(docs, test_qid, 10))
    print("semantic_coverage@10:", round(semantic_coverage(docs, test_qid, 10), 4))
    print("sparse_coverage@10:", round(sparse_query_coverage(docs, test_query, 10), 4))

Test qid: 1117099
Query: what is a active margin

Method: dense
Top docs: ['6122722', '8093934', '1445087', '3349612', '3543106']
nDCG@10: 0.8803
hit_grade3@10: 1
semantic_coverage@10: 0.7344
sparse_coverage@10: 0.8

Method: sparse_bm25
Top docs: ['1445087', '7610604', '8446502', '6177939', '8093934']
nDCG@10: 0.7354
hit_grade3@10: 1
semantic_coverage@10: 0.7261
sparse_coverage@10: 1.0

Method: fusion
Top docs: ['6122722', '8093934', '1445087', '3349612', '8446505']
nDCG@10: 0.7322
hit_grade3@10: 1
semantic_coverage@10: 0.7125
sparse_coverage@10: 1.0

Method: dual_residual
Top docs: ['6122722', '7610604', '8093934', '3349612', '5752877']
nDCG@10: 0.4249
hit_grade3@10: 1
semantic_coverage@10: 0.6206
sparse_coverage@10: 1.0


## 21. Define retrieval families and sweep grids

This notebook now treats **fusion, hybrid MMR, hybrid pricing, and dual residual** as parameterized retrieval families.

### Retrieval families
- **dense**: dense embedding similarity only
- **sparse_bm25**: standard BM25 only
- **fusion**: normalized dense and BM25 scores blended by `alpha`
- **hybrid_mmr**: fused relevance minus an MMR diversity penalty with `lambda_div`
- **hybrid_pricing**: fused relevance minus an average-similarity pricing penalty with `lambda_div`
- **dual_residual**: residualized dense + weighted BM25 retrieval with early stopping

### Parameters to sweep
- **fusion**: `alpha`
- **hybrid_mmr**: `alpha`, `lambda_div`
- **hybrid_pricing**: `alpha`, `lambda_div`
- **dual_residual**: `alpha`, `eta`, `sparse_decay`, `beta_stop`, `min_combined_residual_frac`, `min_combined_score`

The grids below are intentionally easy to edit before running the full experiment.


In [22]:
# =========================
# Parameter grids
# =========================

FUSION_ALPHAS = [0.25, 0.50, 0.75, 0.90]

HYBRID_MMR_ALPHAS = [0.50, 0.75, 0.90]
HYBRID_MMR_LAMBDAS = [0.10, 0.30, 0.50]

HYBRID_PRICING_ALPHAS = [0.50, 0.75, 0.90]
HYBRID_PRICING_LAMBDAS = [0.10, 0.30, 0.50]

DUALRES_ALPHAS = [0.50, 0.75, 0.90]
DUALRES_ETAS = [0.10, 0.20, 0.35, 0.50]
DUALRES_SPARSE_DECAYS = [0.50, 1.00, 2.00]
DUALRES_BETA_STOPS = [0.25, 0.50, 0.75]
DUALRES_MIN_RESIDUALS = [0.05, 0.10]
DUALRES_MIN_SCORES = [0.05, 0.10]

print("Fusion configs:", len(FUSION_ALPHAS))
print("Hybrid MMR configs:", len(HYBRID_MMR_ALPHAS) * len(HYBRID_MMR_LAMBDAS))
print("Hybrid Pricing configs:", len(HYBRID_PRICING_ALPHAS) * len(HYBRID_PRICING_LAMBDAS))
print("Dual residual configs:", len(DUALRES_ALPHAS) * len(DUALRES_ETAS) * len(DUALRES_SPARSE_DECAYS) * len(DUALRES_BETA_STOPS) * len(DUALRES_MIN_RESIDUALS) * len(DUALRES_MIN_SCORES))


Fusion configs: 4
Hybrid MMR configs: 9
Hybrid Pricing configs: 9
Dual residual configs: 432


## 22. Method naming helpers

The helper functions below create readable method names and preserve parameter metadata so later analysis can group by family or compare settings directly.


In [23]:
def fmt_float(x: Optional[float]) -> str:
    if x is None:
        return "none"
    return str(x).replace(".", "")

def fusion_method_name(alpha: float) -> str:
    return f"fusion_a{fmt_float(alpha)}"

def mmr_method_name(alpha: float, lambda_div: float) -> str:
    return f"mmr_a{fmt_float(alpha)}_l{fmt_float(lambda_div)}"

def pricing_method_name(alpha: float, lambda_div: float) -> str:
    return f"pricing_a{fmt_float(alpha)}_l{fmt_float(lambda_div)}"

def dualres_method_name(
    alpha: float,
    eta: float,
    sparse_decay: float,
    beta_stop: float,
    min_combined_residual_frac: float,
    min_combined_score: Optional[float],
) -> str:
    return (
        f"dualres_a{fmt_float(alpha)}"
        f"_eta{fmt_float(eta)}"
        f"_sd{fmt_float(sparse_decay)}"
        f"_b{fmt_float(beta_stop)}"
        f"_r{fmt_float(min_combined_residual_frac)}"
        f"_s{fmt_float(min_combined_score)}"
    )


## 23. Build a unified method registry

Each registry entry stores:
- `method`: the unique config name
- `family`: retrieval family
- `params`: parameter dictionary
- `retriever`: callable used by the evaluation code

This lets the rest of the notebook run one common evaluation pipeline for every method family and every parameter configuration.


In [24]:
def build_method_registry(top_k: int = TOP_K) -> List[Dict]:
    methods = []

    # Baselines
    methods.append({
        "method": "dense",
        "family": "dense",
        "params": {},
        "retriever": lambda query_text, k=top_k: retrieve_dense(query_text, k=k),
    })
    methods.append({
        "method": "sparse_bm25",
        "family": "sparse_bm25",
        "params": {},
        "retriever": lambda query_text, k=top_k: retrieve_sparse(query_text, k=k),
    })

    # Fusion
    for alpha in FUSION_ALPHAS:
        methods.append({
            "method": fusion_method_name(alpha),
            "family": "fusion",
            "params": {"alpha": alpha},
            "retriever": lambda query_text, alpha=alpha, k=top_k: retrieve_fusion(
                query_text, alpha=alpha, k=k
            ),
        })

    # Hybrid MMR
    for alpha, lambda_div in product(HYBRID_MMR_ALPHAS, HYBRID_MMR_LAMBDAS):
        methods.append({
            "method": mmr_method_name(alpha, lambda_div),
            "family": "hybrid_mmr",
            "params": {"alpha": alpha, "lambda_div": lambda_div},
            "retriever": lambda query_text, alpha=alpha, lambda_div=lambda_div, k=top_k: retrieve_hybrid_mmr(
                query_text, alpha=alpha, lambda_div=lambda_div, k=k
            ),
        })

    # Hybrid Pricing
    for alpha, lambda_div in product(HYBRID_PRICING_ALPHAS, HYBRID_PRICING_LAMBDAS):
        methods.append({
            "method": pricing_method_name(alpha, lambda_div),
            "family": "hybrid_pricing",
            "params": {"alpha": alpha, "lambda_div": lambda_div},
            "retriever": lambda query_text, alpha=alpha, lambda_div=lambda_div, k=top_k: retrieve_hybrid_pricing(
                query_text, alpha=alpha, lambda_div=lambda_div, k=k
            ),
        })

    # Dual Residual
    for alpha, eta, sparse_decay, beta_stop, min_resid, min_score in product(
        DUALRES_ALPHAS,
        DUALRES_ETAS,
        DUALRES_SPARSE_DECAYS,
        DUALRES_BETA_STOPS,
        DUALRES_MIN_RESIDUALS,
        DUALRES_MIN_SCORES,
    ):
        methods.append({
            "method": dualres_method_name(
                alpha=alpha,
                eta=eta,
                sparse_decay=sparse_decay,
                beta_stop=beta_stop,
                min_combined_residual_frac=min_resid,
                min_combined_score=min_score,
            ),
            "family": "dual_residual",
            "params": {
                "alpha": alpha,
                "eta": eta,
                "sparse_decay": sparse_decay,
                "beta_stop": beta_stop,
                "min_combined_residual_frac": min_resid,
                "min_combined_score": min_score,
            },
            "retriever": lambda query_text,
                               alpha=alpha,
                               eta=eta,
                               sparse_decay=sparse_decay,
                               beta_stop=beta_stop,
                               min_resid=min_resid,
                               min_score=min_score,
                               k=top_k: retrieve_dual_residual(
                                    query_text,
                                    alpha=alpha,
                                    eta=eta,
                                    sparse_decay=sparse_decay,
                                    beta_stop=beta_stop,
                                    min_combined_residual_frac=min_resid,
                                    min_combined_score=min_score,
                                    k=k,
                               ),
        })

    return methods

method_registry = build_method_registry(top_k=TOP_K)
len(method_registry)


456

## 24. Inspect a few registered methods


In [25]:
for entry in method_registry[:10]:
    print(entry["method"], entry["family"], entry["params"])


dense dense {}
sparse_bm25 sparse_bm25 {}
fusion_a025 fusion {'alpha': 0.25}
fusion_a05 fusion {'alpha': 0.5}
fusion_a075 fusion {'alpha': 0.75}
fusion_a09 fusion {'alpha': 0.9}
mmr_a05_l01 hybrid_mmr {'alpha': 0.5, 'lambda_div': 0.1}
mmr_a05_l03 hybrid_mmr {'alpha': 0.5, 'lambda_div': 0.3}
mmr_a05_l05 hybrid_mmr {'alpha': 0.5, 'lambda_div': 0.5}
mmr_a075_l01 hybrid_mmr {'alpha': 0.75, 'lambda_div': 0.1}


## 25. Unified benchmark runner

The next functions:
1. evaluate each configuration across a chosen set of queries,
2. preserve method-family metadata and parameter columns,
3. return a long-form results table that works for both per-query analysis and aggregate summaries.


In [26]:
def run_method_registry(
    method_registry: List[Dict],
    qids: Optional[List[str]] = None,
    k: int = TOP_K,
    verbose: bool = True,
) -> pd.DataFrame:
    all_rows = []

    for entry in method_registry:
        method_name = entry["method"]
        family = entry["family"]
        params = entry["params"]
        retriever_fn = entry["retriever"]

        df = evaluate_method_over_queries(
            method_name=method_name,
            retrieve_fn=retriever_fn,
            qids=qids,
            k=k,
            verbose=verbose,
        )

        df["family"] = family
        for key, value in params.items():
            df[key] = value

        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

def mean_ignore_none(series: pd.Series) -> float:
    vals = pd.to_numeric(series, errors="coerce")
    vals = vals.dropna()
    return float(vals.mean()) if len(vals) else np.nan

def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "nDCG@k",
        "irrelevant@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
    ]

    summary = (
        results_df
        .groupby(["method", "family"], as_index=False)[metric_cols]
        .mean()
    )

    for thr in SEMANTIC_THRESHOLDS:
        col = f"docs_to_semantic_{thr:.2f}"
        extra = (
            results_df.groupby(["method", "family"])[col]
            .apply(mean_ignore_none)
            .reset_index(name=col)
        )
        summary = summary.merge(extra, on=["method", "family"], how="left")

    for thr in SPARSE_THRESHOLDS:
        col = f"docs_to_sparse_{thr:.2f}"
        extra = (
            results_df.groupby(["method", "family"])[col]
            .apply(mean_ignore_none)
            .reset_index(name=col)
        )
        summary = summary.merge(extra, on=["method", "family"], how="left")

    # attach one representative parameter row per method
    param_cols = [
        c for c in results_df.columns
        if c not in {
            "qid", "method", "family", "k",
            "nDCG@k", "irrelevant@k", "redundancy@k", "hit_grade3@k",
            "semantic_coverage@k", "sparse_coverage@k", "docs_retrieved",
            *[f"docs_to_semantic_{thr:.2f}" for thr in SEMANTIC_THRESHOLDS],
            *[f"docs_to_sparse_{thr:.2f}" for thr in SPARSE_THRESHOLDS],
        }
    ]
    if param_cols:
        params_df = results_df[["method", "family"] + param_cols].drop_duplicates(subset=["method", "family"])
        summary = summary.merge(params_df, on=["method", "family"], how="left")

    return summary


## 26. Choose a run mode

A full sweep can be large. Start with a pilot run on a subset of queries, then run the full experiment once the code and metrics look correct.


In [27]:
PILOT_NUM_QUERIES = 20
pilot_qids = list(queries_frozen.keys())[:PILOT_NUM_QUERIES]
full_qids = list(queries_frozen.keys())

print("Pilot queries:", len(pilot_qids))
print("Full queries:", len(full_qids))
print("Total method configs:", len(method_registry))


Pilot queries: 20
Full queries: 97
Total method configs: 456


## 27. Pilot sweep

This runs all retrieval families and all configured parameter sweeps on a smaller set of queries first.


In [28]:
# Uncomment to run the pilot sweep.
# pilot_results_df = run_method_registry(method_registry, qids=pilot_qids, k=TOP_K, verbose=True)
# pilot_results_path = os.path.join(CACHE_DIR, "pilot_results_all_methods.csv")
# pilot_results_df.to_csv(pilot_results_path, index=False)
# pilot_summary_df = summarize_results(pilot_results_df)
# pilot_summary_df.sort_values(["family", "nDCG@k"], ascending=[True, False]).head(30)


## 28. Full sweep across all configured retrieval methods

Uncomment the next cell when ready to run the full benchmark.


In [ ]:
results_df = run_method_registry(method_registry, qids=full_qids, k=TOP_K, verbose=True)
results_path = os.path.join(CACHE_DIR, "results_all_methods_swept.csv")
results_df.to_csv(results_path, index=False)
print("Saved full results to:", results_path)
results_df.head()


Evaluating pricing_a09_l01:  62%|█████████▎     | 60/97 [01:16<00:47,  1.28s/it]

## 29. Load an existing run if needed

Use this if you already ran the sweep and saved the long-form results table.


In [ ]:
# results_df = pd.read_csv(os.path.join(CACHE_DIR, "results_all_methods_swept.csv"))
# results_df.head()


## 30. Aggregate method-level results

This produces one row per method configuration with all metrics:
- ranking quality
- irrelevance
- redundancy
- grade-3 hit rate
- semantic coverage
- sparse coverage
- efficiency thresholds


In [ ]:
summary_df = summarize_results(results_df)
summary_df = summary_df.sort_values("nDCG@k", ascending=False).reset_index(drop=True)
summary_df.round(4).head(25)


## 31. Best configuration per family

This is often the most useful paper-facing summary. You can choose the criterion depending on the story you want to tell.

Below, the default criterion is:
1. highest `nDCG@k`
2. then highest `semantic_coverage@k`
3. then lowest `redundancy@k`


In [ ]:
def best_config_per_family(
    summary_df: pd.DataFrame,
    sort_cols: Optional[List[str]] = None,
    ascending: Optional[List[bool]] = None,
) -> pd.DataFrame:
    if sort_cols is None:
        sort_cols = ["nDCG@k", "semantic_coverage@k", "redundancy@k"]
    if ascending is None:
        ascending = [False, False, True]

    pieces = []
    for family, group in summary_df.groupby("family"):
        best = group.sort_values(sort_cols, ascending=ascending).head(1)
        pieces.append(best)
    return pd.concat(pieces, ignore_index=True)

best_family_df = best_config_per_family(summary_df)
best_family_df.round(4)


## 32. Alternative best-config views

Sometimes you want the best configuration per family under a different objective, such as:
- highest semantic coverage
- highest grade-3 hit rate
- best low-redundancy coverage tradeoff


In [ ]:
def best_by_metric(summary_df: pd.DataFrame, metric: str, higher_is_better: bool = True) -> pd.DataFrame:
    ascending = not higher_is_better
    pieces = []
    for family, group in summary_df.groupby("family"):
        best = group.sort_values(metric, ascending=ascending).head(1)
        pieces.append(best)
    return pd.concat(pieces, ignore_index=True)

best_semantic_df = best_by_metric(summary_df, "semantic_coverage@k", higher_is_better=True)
best_hit3_df = best_by_metric(summary_df, "hit_grade3@k", higher_is_better=True)
best_low_redundancy_df = best_by_metric(summary_df, "redundancy@k", higher_is_better=False)


## 33. Family-level sweep summaries

These compact summaries are helpful for seeing what the parameter sweeps are doing before inspecting individual configurations.


In [ ]:
def family_summary_tables(results_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out = {}

    if "alpha" in results_df.columns:
        out["alpha_by_family"] = (
            results_df.dropna(subset=["alpha"])
            .groupby(["family", "alpha"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k", "sparse_coverage@k"]]
            .mean()
            .reset_index()
            .sort_values(["family", "alpha"])
        )

    if {"family", "lambda_div"}.issubset(results_df.columns):
        out["lambda_div_by_family"] = (
            results_df[results_df["family"].isin(["hybrid_mmr", "hybrid_pricing"])]
            .dropna(subset=["lambda_div"])
            .groupby(["family", "alpha", "lambda_div"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k"]]
            .mean()
            .reset_index()
            .sort_values(["family", "alpha", "lambda_div"])
        )

    if "eta" in results_df.columns:
        out["dualres_eta"] = (
            results_df[results_df["family"] == "dual_residual"]
            .dropna(subset=["eta"])
            .groupby(["eta"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k", "docs_retrieved"]]
            .mean()
            .reset_index()
            .sort_values("eta")
        )

    return out

family_tables = family_summary_tables(results_df)
for name, table in family_tables.items():
    print("\n", name)
    display(table.head(20))


## 34. Visualize the tradeoffs

Two especially useful plots:
- `nDCG@k` versus `redundancy@k`
- `semantic_coverage@k` versus `docs_retrieved`

The first shows ranking-diversity tradeoffs.  
The second shows coverage-efficiency tradeoffs.


In [ ]:
def plot_tradeoff(
    summary_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    label_points: bool = False,
):
    plt.figure(figsize=(8, 6))
    families = summary_df["family"].unique()

    for family in families:
        sub = summary_df[summary_df["family"] == family]
        plt.scatter(sub[x_col], sub[y_col], label=family, alpha=0.75)
        if label_points:
            for _, row in sub.iterrows():
                plt.annotate(row["method"], (row[x_col], row[y_col]), fontsize=7, alpha=0.7)

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_tradeoff(summary_df, "redundancy@k", "nDCG@k", "nDCG vs redundancy across all configs", label_points=False)
plot_tradeoff(summary_df, "docs_retrieved", "semantic_coverage@k", "Semantic coverage vs docs retrieved", label_points=False)


## 35. Export paper-facing tables

This exports:
- the long-form per-query results table
- the aggregate summary over every method configuration
- the best configuration per family


In [ ]:
# export_dir = CACHE_DIR
# long_results_path = os.path.join(export_dir, "results_all_methods_swept.csv")
# summary_results_path = os.path.join(export_dir, "summary_all_methods_swept.csv")
# best_family_path = os.path.join(export_dir, "best_config_per_family.csv")

# results_df.to_csv(long_results_path, index=False)
# summary_df.to_csv(summary_results_path, index=False)
# best_family_df.to_csv(best_family_path, index=False)

# print("Saved long-form results to:", long_results_path)
# print("Saved summary results to:", summary_results_path)
# print("Saved best-per-family table to:", best_family_path)


## 36. Minimal run checklist

1. Run the dataset-loading and candidate-pool cells.
2. Build BM25 and dense embeddings.
3. Run a quick spot-check on one query.
4. Edit the sweep grids if needed.
5. Run the pilot sweep first.
6. Inspect `pilot_summary_df` and the family-level tables.
7. Run the full sweep.
8. Export `results_df`, `summary_df`, and `best_family_df`.

This notebook is now structured so the same evaluation pipeline works for:
- clearly defined retrieval methods
- parameter sweeps for fusion, hybrid MMR, hybrid pricing, and dual residual
- ranking, redundancy, coverage, and efficiency metrics
